# 02 · Isaac ROS 5: accelerate a real perception node
**15 minutes.** Objective: launch NVIDIA's real CUDA AprilTag component and distinguish **a loaded detector** from **a measured detection**.

ROS 2 supplies the typed graph. Isaac ROS supplies accelerated implementations and negotiated transports. A CPU-published image entering a CUDA node is **not automatically GPU zero-copy**.

There is deliberately **no camera source yet**. Notebook03 supplies a real rendered onboard camera after starting Isaac Sim. This notebook starts the detector alone, not the gateway that requires a running native simulator. We never fabricate tags or copy a recorded detection into a live result.

In [ ]:
from pathlib import Path
import os, sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the host Python 3.12 kernel created by scripts/notebook_setup.sh")
# This kernel is deliberately not a ROS / Kit interpreter.
start = Path(os.environ.get("WORKSHOP_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/workshop.py").is_file()
             and (p / "src/microduck_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set WORKSHOP_ROOT")
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import Session
session = Session(ROOT, work=os.environ.get("WORKSHOP_NOTEBOOK_WORK"))
VALIDATE = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "validate"
print("Host Python:", sys.version.split()[0])
print("Mode:", "safe validation (hardware SKIPPED)" if VALIDATE else "interactive")

## Predict · 2 minutes
Can `ros2 node list` succeed while the detector produces no detections? Which two input topics should a calibrated image detector need?

The first live cell requires Notebook01's actual tutorial checkpoint. Review the NVIDIA software terms with staff. Opting in below acknowledges that you reviewed and accepted those terms; it must not be prefilled by the instructor.

In [ ]:
RUN_GPU = False                  # explicitly change to True when your GPU is allocated
ACCEPT_NVIDIA_EULA = False        # only after reviewing the terms
GPU = os.environ.get("WORKSHOP_GPU", "0")
ROS_IMAGE = os.environ.get("WORKSHOP_ROS_IMAGE", "roscon/microduck-ros5:local")
NODE = "/workshop/microduck/apriltag"

In [ ]:
def start_detector():
    session.require("01")
    session.gpu_preflight(GPU, accept_eula=ACCEPT_NVIDIA_EULA)
    session.start("detector", ROS_IMAGE,
        ["ros2", "launch", "microduck_workshop_ros", "apriltag.launch.py", "tag_size:=0.20"],
        options=session.gpu_options(GPU))
    session.wait_ros_node("detector", NODE, timeout=60)
session.step("launch isolated CUDA AprilTag node (no image source)", RUN_GPU, start_detector)

## Observe · 4 minutes
Inspect the real graph and parameters. Locate the image and CameraInfo subscriptions, their types, the output and the configured `CUDA` backend. Graph discovery alone cannot prove the CUDA kernels ran on an image: no input has arrived yet.

In [ ]:
def inspect_detector():
    session.ros("detector", ["node", "info", NODE])
    session.ros("detector", ["param", "get", NODE, "backends"])
    session.ros("detector", ["param", "get", NODE, "tag_family"])
    session.ros("detector", ["param", "get", NODE, "size"])
    session.ros("detector", ["topic", "list", "-t"])
session.step("inspect real detector interfaces", RUN_GPU, inspect_detector)

## Task · 4 minutes
Change `INSPECT_TOPIC` to the camera-info input, rerun the cell, and describe the publisher count. Predict what happens to inferred tag range if physical tag size is wrong. **Do not claim a detection before an actual image was processed.**

Tag family identifies the code family; tag size is the physical edge length used for pose/range. Image calibration, optical frame and capture identity must describe the actual pixels. In our teaching scene the authored tag is 0.20 m; this is not hardware calibration.

In [ ]:
INSPECT_TOPIC = "/workshop/microduck/camera/image_rect"
session.step("inspect input endpoint count", RUN_GPU,
             lambda: session.ros("detector", ["topic", "info", INSPECT_TOPIC, "--verbose"]))

## Debug · 3 minutes
- Missing CUDA/library errors: keep the actual container log and ask staff to verify the pinned image/driver. Do not switch to invented results.
- Node discovery deadline: inspect the real launch log; a running Docker container is not a loaded component.
- Zero publishers: expected **here**, because the camera first exists in Notebook03. Empty detections with a running camera are a different diagnosis.
- NITROS negotiation messages do not by themselves prove end-to-end GPU zero-copy.

**Checkpoint:** identify the two inputs and say what is still unverified. The receipt below says `DETECTOR_ONLY_NO_CAMERA`, not perception accepted.

## Cleanup · 2 minutes
Stop only this isolated detector. Notebook03 explicitly launches a fresh complete ROS bridge **after** the native simulator becomes ready; it does not reuse this no-input container.

In [ ]:
def finish_detector():
    session.wait_ros_node("detector", NODE, timeout=30)
    result = session.ros("detector", ["param", "get", NODE, "backends"])
    if "CUDA" not in result.stdout:
        raise RuntimeError("Actual detector backend is not CUDA; no completion checkpoint")
    session.complete("02", {"kind": "DETECTOR_ONLY_NO_CAMERA", "node": NODE,
                            "tag_size_m": 0.20, "input_supplied": False})
    session.stop("detector")
session.step("save limited checkpoint and stop owned detector", RUN_GPU, finish_detector)